<a href="https://colab.research.google.com/github/B-21-g-23/AquaCropvisb/blob/main/sentinel_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORT LIBRARIES
import ee
import geemap
!pip install xee
import xee
import xarray as xr
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.3/479.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: earthengine-api
    Found existing installation: earthengine-api 1.5.24
    Uninstalling earthengine-api-1.5.24:
      Successfully uninstalled earthengine-api-1.5.24


In [ ]:
 #Colab: install Earth Engine API
!pip install --upgrade earthengine-api geemap

import ee
import geemap

# Trigger OAuth2 authentication flow
ee.Authenticate()
ee.Initialize(
    project = 'ee-birukgetaneh13',
    opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
val = img.reduceRegion(
    reducer = ee.Reducer.mean(),
    geometry = point,
    scale = 30
)
return ee.Feature(None, {
    'date': img.date().format(),
    'VV_dB': val.get('VV'),   # <-- this line ensures VV backscatter is saved
    'point_id': point_id
})


NameError: name 'img' is not defined

In [ ]:
# Step 1: Initialize Earth Engine
import ee
ee.Authenticate()
ee.Initialize(project='ee-birukgetaneh13', opt_url='https://earthengine-highvolume.googleapis.com')

# Step 2: Define your GPS points (lon, lat)
coords = [
    [39.36977, 9.4961],
    [39.37016, 9.49667],
    [39.36961, 9.496833],
    [39.36999, 9.497267],
    [39.37002, 9.497728],
    [39.37289, 9.501017],
    [39.37345, 9.501523],
    [39.37399, 9.501749],
    [39.37391, 9.502328],
    [39.37457, 9.502607],
    [39.3777, 9.498027],
    [39.37862, 9.497881],
    [39.37906, 9.497156],
    [39.3778, 9.49711],
    [39.37717, 9.496894]
]

points = [ee.Geometry.Point(lon, lat) for lon, lat in coords]

# Step 3: Load Sentinel-1 VV backscatter data
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterDate('2024-01-01', '2024-05-30')  # Wheat season 2024
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .select('VV'))

# Step 4: Function to extract VV backscatter for each point
def extract_point_timeseries(point, point_id):
    def get_value(img):
        val = img.reduceRegion(
            reducer = ee.Reducer.mean(),
            geometry = point,
            scale = 30
        )
        return ee.Feature(None, {
            'date': img.date().format(),
            'VV_dB': val.get('VV'),   # backscatter value
            'point_id': point_id
        })
    return s1.map(get_value)

# Step 5: Apply extraction for all points
features = ee.FeatureCollection([])
for i, pt in enumerate(points):
    fc = extract_point_timeseries(pt, f'Point_{i+1}')
    features = features.merge(fc)

# Step 6: Export results as CSV to Google Drive
task = ee.batch.Export.table.toDrive(
    collection = features,
    description = 'Sentinel1_SoilMoisturePoints_Wheat2024',
    folder = 'sentinel-1 SM',
    fileNamePrefix = 'sentinel1_soilmoisture_points',
    fileFormat = 'CSV'
)

task.start()

# Step 7: Monitor export
import time
while task.active():
    print('Exporting soil moisture proxy time-series...')
    time.sleep(30)

print('Export finished:', task.status())

Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting 

Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Exporting soil moisture proxy time-series...
Export finished: {'state': 'COMPLETED', 'description': 'Sentinel1_SoilMoisturePoints_Wheat2024', 'priority': 100, 'creation_timestamp_ms': 1775888759092, 'update_timestamp_ms': 1775892540115, 'start_timestamp_ms': 1775888768738, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1D34bZMi

In [ ]:
def extract_point_timeseries(point, point_id):
    def get_value(img):
        val = img.reduceRegion(
            reducer = ee.Reducer.mean(),
            geometry = point,
            scale = 30
        )
        return ee.Feature(None, {
            'date': img.date().format(),
            'VV_dB': val.get('VV'),   # <-- ensures VV backscatter is included
            'point_id': point_id
        })
    return s1.map(get_value)


In [ ]:
point_buffer = point.buffer(50)  # 50 m buffer


NameError: name 'point' is not defined